# Làm sạch số học & cân bằng dữ liệu — Camera AI ONT

Notebook này nhận đầu vào từ `preprocessing_data.ipynb` (`ids_dataset_labeled.csv`) và thực hiện:

| Phần | Nội dung |
|---|---|
| **1** | Kiểm tra chất lượng số học: NaN, Inf, **giá trị giả vô cực**, giá trị âm vô lý |
| **2** | Xử lý các vấn đề tìm được |
| **3** | Khử trùng lặp (**trước** khi split, nếu không sẽ rò rỉ) |
| **4** | Chia train / val / test theo tỉ lệ phân tầng |
| **5** | Điền NaN — tham số **chỉ học từ tập train** |
| **6** | Chuẩn hoá thang đo về [0, 1] — tham số cũng **chỉ học từ train** |
| **7** | Cân bằng lớp: so sánh 5 chiến lược bằng kiểm định lặp |
| **8** | Lưu kết quả |

> **Thứ tự các bước không tuỳ tiện.** Khử trùng lặp phải đứng trước split; còn điền NaN, chuẩn hoá
> và cân bằng phải đứng *sau* split, với tham số **chỉ học từ tập train**. Làm ngược lại là rò rỉ
> thông tin từ tập test vào quá trình huấn luyện, khiến điểm số đẹp giả tạo.

In [1]:
# Cấu hình số luồng CPU trước khi import NumPy/Pandas.
import os

logical_cpus = os.cpu_count() or 1
compute_threads = max(1, logical_cpus // 2)

os.environ['OPENBLAS_NUM_THREADS'] = str(compute_threads)
os.environ['MKL_NUM_THREADS'] = str(compute_threads)
os.environ['OMP_NUM_THREADS'] = str(compute_threads)
os.environ['NUMEXPR_NUM_THREADS'] = str(compute_threads)

print(f'CPU logic: {logical_cpus} | BLAS: {compute_threads} luồng')

CPU logic: 12 | BLAS: 6 luồng


In [2]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier

from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.pipeline import Pipeline as SkPipeline

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

RANDOM_STATE = 42

BASE_DIR    = Path.cwd().parent if Path.cwd().name == 'Decesion_trees_train_model' else Path.cwd()
DATASET_DIR = BASE_DIR / 'dataset_csv'

IN_CSV   = DATASET_DIR / 'ids_dataset_labeled.csv'
META_IN  = DATASET_DIR / 'feature_meta.json'
OUT_CSV  = DATASET_DIR / 'ids_dataset_ready.csv'
META_OUT = DATASET_DIR / 'preprocessing_meta.json'

# Tham số chuẩn hoá là thứ CẦN KHI TRIỂN KHAI, không phải sản phẩm trung gian,
# nên ghi thẳng vào thư mục xuất mô hình thay vì để lẫn trong dataset_csv.
EXPORT_DIR = Path.cwd() / 'model_export'
EXPORT_DIR.mkdir(exist_ok=True)

assert IN_CSV.exists(),  f'Chưa có {IN_CSV} — chạy preprocessing_data.ipynb trước.'
assert META_IN.exists(), f'Chưa có {META_IN} — chạy preprocessing_data.ipynb trước.'

meta_in  = json.loads(META_IN.read_text(encoding='utf-8'))
FEATURES = meta_in['features']
df       = pd.read_csv(IN_CSV)

print(f'Đầu vào : {IN_CSV.name}  -> {df.shape[0]} flow x {df.shape[1]} cột')
print(f'Feature : {len(FEATURES)}')
print(f'Nhãn    : {df["label"].value_counts().to_dict()}')

Đầu vào : ids_dataset_labeled.csv  -> 6024 flow x 34 cột
Feature : 24
Nhãn    : {'normal': 3204, 'scanport': 2820}


---
## PHẦN 1 — Kiểm tra chất lượng số học

Không sửa gì ở phần này, chỉ đo đạc. Mọi thao tác ở Phần 2 đều phải dựa trên số liệu ở đây.

In [3]:
# =========================================================
# 1.1 — NaN và Inf theo đúng nghĩa đen
# =========================================================
Xnum = df[FEATURES]

nan_per_col = Xnum.isna().sum()
inf_per_col = pd.Series(np.isinf(Xnum.to_numpy()).sum(axis=0), index=FEATURES)

print(f'Tổng ô NaN : {nan_per_col.sum()}')
print(f'Tổng ô Inf : {inf_per_col.sum()}')

problem = pd.DataFrame({'NaN': nan_per_col, 'Inf': inf_per_col})
problem = problem[(problem['NaN'] > 0) | (problem['Inf'] > 0)]

if len(problem):
    print('\nCột có vấn đề:')
    print(problem.to_string())
else:
    print('\nKhông có NaN/Inf theo nghĩa đen.')
    print('=> Nhưng KHÔNG có nghĩa là dữ liệu sạch — xem ô tiếp theo.')

Tổng ô NaN : 0
Tổng ô Inf : 0

Không có NaN/Inf theo nghĩa đen.
=> Nhưng KHÔNG có nghĩa là dữ liệu sạch — xem ô tiếp theo.


### 1.2 — Giá trị giả vô cực (vấn đề thật của dataset này)

Bộ chuyển đổi `pcap_to_csv.py` tính tốc độ như sau (dòng 259 và 276):

```python
safe_dur = dur if dur > 0 else 1e-6          # chặn chia cho 0
...
round(tot_pkts  / safe_dur, 6),              # flow_pkts_per_s
round(tot_bytes / safe_dur, 6),              # flow_bytes_per_s
```

Với flow chỉ có một gói (hoặc nhiều gói trùng dấu thời gian), `duration = 0` nên mẫu số thành
`1e-6`, tức **mọi tốc độ bị nhân lên một triệu lần**. Kết quả là các giá trị bất khả thi về mặt vật
lý — tới **4,19 GB/s** trên mạng LAN — nhưng lại trông như số thực bình thường nên `isna()` và
`isinf()` đều không bắt được.

Đây thực chất là **vô cực được mã hoá thành một con số hữu hạn**. Tốc độ của một flow có
`duration = 0` là **không xác định**, không phải "rất lớn".

Dùng **hai tiêu chí bổ sung cho nhau** để bắt hết:

1. **Dấu vết của hằng số chặn.** Khi `duration = 0` thì `flow_pkts_per_s` đúng bằng `tot_pkts × 1e6`.
   Nhận biết chính xác tuyệt đối.
2. **Ngưỡng vật lý.** Tiêu chí trên bỏ sót các flow có duration *dương* nhưng nhỏ hơn một
   micro-giây (cột `duration` trong CSV bị làm tròn 6 chữ số, còn phép chia lại dùng giá trị chưa
   làm tròn, nên không khớp công thức). Với chúng, ta dựa vào giới hạn vật lý của đường truyền
   1 Gbps: không flow nào vượt quá **125 MB/s** hay **1,49 triệu gói/giây** được.

In [4]:
# =========================================================
# 1.2 — Phát hiện giá trị giả vô cực
# =========================================================
DIV_GUARD = 1e-6                       # hằng số chặn chia 0 trong pcap_to_csv.py
RATE_FEATURES = [f for f in FEATURES if f.endswith('_per_s')]

# Giới hạn vật lý của đường truyền Ethernet 1 Gbps
MAX_BYTES_PER_S = 125_000_000          # 1 Gbps
MAX_PKTS_PER_S  = 1_488_095            # khung nhỏ nhất 64 byte ở tốc độ 1 Gbps

# (1) dấu vết hằng số chặn: duration = 0  <=>  flow_pkts_per_s == tot_pkts / 1e-6
is_guard = pd.Series(
    np.isclose(df['flow_pkts_per_s'], df['tot_pkts'] / DIV_GUARD, rtol=1e-6),
    index=df.index)

# (2) vượt giới hạn vật lý
is_impossible = (df['flow_bytes_per_s'] > MAX_BYTES_PER_S) | (df['flow_pkts_per_s'] > MAX_PKTS_PER_S)

is_pseudo_inf = (is_guard | is_impossible).rename('rate_undefined')

print(f'Cột tốc độ bị ảnh hưởng      : {RATE_FEATURES}\n')
print(f'(1) Dính hằng số chặn (duration=0)     : {is_guard.sum():>5}')
print(f'(2) Vượt giới hạn vật lý 1 Gbps        : {is_impossible.sum():>5}')
print(f'    trong đó KHÔNG thuộc nhóm (1)      : {(is_impossible & ~is_guard).sum():>5}  '
      f'<- duration dương nhưng dưới 1 micro-giây')
print(f'{"":->44}')
print(f'Tổng flow có tốc độ không đáng tin     : {is_pseudo_inf.sum():>5} / {len(df)} '
      f'({is_pseudo_inf.mean():.1%})\n')

print('Giá trị lớn nhất của các cột tốc độ:')
for f in RATE_FEATURES:
    print(f'  {f:18s} toàn bộ = {df[f].max():>17,.1f}   '
          f'khi loại các flow trên = {df.loc[~is_pseudo_inf, f].max():>12,.1f}')

print('\nPhân bố theo lớp:')
print(pd.crosstab(is_pseudo_inf, df['label'],
                  rownames=['tốc độ không xác định'], colnames=['nhãn']).to_string())

Cột tốc độ bị ảnh hưởng      : ['flow_pkts_per_s', 'bwd_pkts_per_s', 'flow_bytes_per_s']

(1) Dính hằng số chặn (duration=0)     :  1075
(2) Vượt giới hạn vật lý 1 Gbps        :    56
    trong đó KHÔNG thuộc nhóm (1)      :    11  <- duration dương nhưng dưới 1 micro-giây
--------------------------------------------
Tổng flow có tốc độ không đáng tin     :  1086 / 6024 (18.0%)

Giá trị lớn nhất của các cột tốc độ:
  flow_pkts_per_s    toàn bộ =       5,242,880.0   khi loại các flow trên =     19,784.5
  bwd_pkts_per_s     toàn bộ =           9,892.2   khi loại các flow trên =      9,892.2
  flow_bytes_per_s   toàn bộ =   4,194,304,000.0   khi loại các flow trên =  9,549,678.9

Phân bố theo lớp:
nhãn                   normal  scanport
tốc độ không xác định                  
False                    2429      2509
True                      775       311


In [5]:
# =========================================================
# 1.3 — Giá trị âm vô lý và giá trị cực đoan
# =========================================================
# IAT (khoảng cách giữa 2 gói) không thể âm -> âm = gói bắt được lệch thứ tự
NON_NEGATIVE = [f for f in FEATURES
                if any(k in f for k in ('iat', 'len', 'cnt', 'bytes', 'pkts', 'ratio', 'std', 'idle'))]

neg_report = {f: int((df[f] < 0).sum()) for f in NON_NEGATIVE if (df[f] < 0).any()}
print('Cột có giá trị âm bất hợp lý:')
if neg_report:
    for f, n in neg_report.items():
        print(f'  {f:18s} {n:>3} flow, nhỏ nhất = {df[f].min():.6f}s  (gói bắt lệch thứ tự)')
else:
    print('  không có')

print('\nĐộ lệch đuôi phân phối (max / phân vị 99):')
tail = []
for f in FEATURES:
    p99 = df[f].quantile(0.99)
    if p99 > 0:
        tail.append({'feature': f, 'p99': round(p99, 2), 'max': round(df[f].max(), 2),
                     'max/p99': round(df[f].max() / p99, 1)})
tail = pd.DataFrame(tail).sort_values('max/p99', ascending=False)
print(tail.head(8).to_string(index=False))

Cột có giá trị âm bất hợp lý:
  flow_iat_min         9 flow, nhỏ nhất = -0.000842s  (gói bắt lệch thứ tự)
  fwd_iat_min          1 flow, nhỏ nhất = -0.000842s  (gói bắt lệch thứ tự)

Độ lệch đuôi phân phối (max / phân vị 99):
              feature          p99          max  max/p99
             tot_pkts        11.77 5.846060e+05  49669.2
bwd_pkts_with_payload         3.00 9.190000e+02    306.3
    bwd_payload_bytes      4096.00 7.008620e+05    171.1
     flow_bytes_per_s 119000000.00 4.194304e+09     35.2
          fwd_iat_min         3.00 9.628000e+01     32.1
          pkt_len_max      1966.18 3.481800e+04     17.7
      fwd_pkt_len_std       666.67 1.131035e+04     17.0
          pkt_len_std       717.15 6.726210e+03      9.4


---
## PHẦN 2 — Xử lý

Ba thao tác, tất cả đều **xác định trước, không học tham số từ dữ liệu**, nên chạy trước khi split
là an toàn (không gây rò rỉ):

| # | Thao tác | Lý do |
|---|---|---|
| 1 | `±Inf` → `NaN` | Phòng thủ cho các lần capture sau; hiện tại dataset chưa có |
| 2 | Tốc độ không đáng tin → `NaN` | Tốc độ đó **không xác định** chứ không phải "rất lớn". Chuyển thành NaN nói đúng sự thật thay vì giấu một con số 4 tỉ bất khả thi |
| 3 | IAT âm → `0` | Khoảng cách giữa hai gói không thể âm; đây là lỗi thứ tự gói khi bắt |

**Không** winsorize / cắt đuôi phân phối: Decision Tree chia theo ngưỡng nên **bất biến với mọi phép
biến đổi đơn điệu** — cắt đuôi không đổi kết quả mà chỉ làm mất thông tin. (Nếu sau này thử hồi quy
logistic, KNN hay mạng nơ-ron thì mới cần, vì các mô hình đó nhạy với thang đo.)


In [6]:
# =========================================================
# 2.1 — Áp dụng làm sạch
# =========================================================
ADD_ZERO_DURATION_FLAG = False    # True = thêm cột cờ (25 feature) | False = giữ đúng 24 feature Excel

clean = df.copy()
log = {}

# (1) Inf -> NaN
n_inf = int(np.isinf(clean[FEATURES].to_numpy()).sum())
clean[FEATURES] = clean[FEATURES].replace([np.inf, -np.inf], np.nan)
log['inf_to_nan'] = n_inf

# (2) tốc độ không đáng tin -> NaN
clean.loc[is_pseudo_inf.values, RATE_FEATURES] = np.nan
log['pseudo_inf_to_nan'] = int(is_pseudo_inf.sum()) * len(RATE_FEATURES)

# (3) giá trị âm bất hợp lý -> 0
n_neg = 0
for f in NON_NEGATIVE:
    mask = clean[f] < 0
    n_neg += int(mask.sum())
    clean.loc[mask, f] = 0.0
log['negative_clipped'] = n_neg

# cột cờ luôn được tính (để ô 2.2 đo được), nhưng chỉ vào feature set nếu bật công tắc
clean['zero_duration_flow'] = is_pseudo_inf.astype(int).values
FEATURES_V2 = FEATURES + (['zero_duration_flow'] if ADD_ZERO_DURATION_FLAG else [])

print(f'(1) Inf -> NaN                        : {log["inf_to_nan"]:>5} ô')
print(f'(2) Tốc độ không xác định -> NaN      : {log["pseudo_inf_to_nan"]:>5} ô '
      f'({is_pseudo_inf.sum()} flow x {len(RATE_FEATURES)} cột)')
print(f'(3) Giá trị âm -> 0                   : {log["negative_clipped"]:>5} ô')
print(f'\nThêm cột cờ zero_duration_flow : {ADD_ZERO_DURATION_FLAG}')
print(f'Số feature dùng để train       : {len(FEATURES_V2)}')
print(f'NaN cần điền ở Phần 5          : {int(clean[FEATURES_V2].isna().sum().sum())} ô')

(1) Inf -> NaN                        :     0 ô
(2) Tốc độ không xác định -> NaN      :  3258 ô (1086 flow x 3 cột)
(3) Giá trị âm -> 0                   :    10 ô

Thêm cột cờ zero_duration_flow : False
Số feature dùng để train       : 24
NaN cần điền ở Phần 5          : 3258 ô


In [7]:
# =========================================================
# 2.2 — Cột cờ có đáng thêm không? Đo thay vì đoán.
# =========================================================
_probe = clean.copy()
_probe[FEATURES] = _probe[FEATURES].fillna(_probe[FEATURES].median())
_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=RANDOM_STATE)

print('So sánh trên 20 phép đo (f1_macro):\n')
_scores = {}
for _tag, _cols in [('24 feature (không cờ)', FEATURES),
                    ('25 feature (có cờ)   ', FEATURES + ['zero_duration_flow'])]:
    _d = _probe.drop_duplicates(subset=_cols + ['label'])
    _s = cross_val_score(DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
                         _d[_cols], _d['label'].map({'normal': 0, 'scanport': 1}),
                         cv=_cv, scoring='f1_macro', n_jobs=-1)
    _scores[_tag] = s_mean = _s.mean()
    print(f'  {_tag}  f1 = {_s.mean():.4f} +- {_s.std():.4f}')

_delta = abs(list(_scores.values())[1] - list(_scores.values())[0])
print(f'\nChênh lệch: {_delta:.4f}  -> {"KHÔNG đáng kể" if _delta < 0.002 else "đáng kể"}')
print('\nKết luận: cột cờ không cải thiện gì. Thông tin "flow gần như tức thời" đã')
print('nằm sẵn trong tot_pkts (flow duration=0 hầu hết chỉ có 1 gói), nên cột cờ bị trùng lặp.')
print(f'=> Giữ đúng {len(FEATURES)} feature từ Excel, ADD_ZERO_DURATION_FLAG = False.')

So sánh trên 20 phép đo (f1_macro):



  24 feature (không cờ)  f1 = 0.9909 +- 0.0025
  25 feature (có cờ)     f1 = 0.9912 +- 0.0032

Chênh lệch: 0.0003  -> KHÔNG đáng kể

Kết luận: cột cờ không cải thiện gì. Thông tin "flow gần như tức thời" đã
nằm sẵn trong tot_pkts (flow duration=0 hầu hết chỉ có 1 gói), nên cột cờ bị trùng lặp.
=> Giữ đúng 24 feature từ Excel, ADD_ZERO_DURATION_FLAG = False.


In [8]:
# =========================================================
# 2.2 — Xác nhận kết quả
# =========================================================
print('Giá trị lớn nhất của các cột tốc độ, trước và sau khi làm sạch:')
for f in RATE_FEATURES:
    before, after = df[f].max(), clean[f].max()
    print(f'  {f:18s} {before:>17,.1f}  ->  {after:>13,.1f}   (giảm {before/max(after,1e-9):>8,.0f} lần)')
print('\n=> Các giá trị bất khả thi về mặt vật lý đã biến mất.')

print(f'\nCòn giá trị âm : {sum(int((clean[f] < 0).sum()) for f in NON_NEGATIVE)}')
print(f'Còn Inf        : {int(np.isinf(clean[FEATURES_V2].fillna(0).to_numpy()).sum())}')
print(f'NaN hiện có    : {int(clean[FEATURES_V2].isna().sum().sum())} (chủ ý tạo ra, sẽ điền ở Phần 5)')

print('\nNaN theo từng cột:')
nan_now = clean[FEATURES_V2].isna().sum()
print(nan_now[nan_now > 0].to_string())

Giá trị lớn nhất của các cột tốc độ, trước và sau khi làm sạch:
  flow_pkts_per_s          5,242,880.0  ->       19,784.5   (giảm      265 lần)
  bwd_pkts_per_s               9,892.2  ->        9,892.2   (giảm        1 lần)
  flow_bytes_per_s     4,194,304,000.0  ->    9,549,678.9   (giảm      439 lần)

=> Các giá trị bất khả thi về mặt vật lý đã biến mất.

Còn giá trị âm : 0
Còn Inf        : 0
NaN hiện có    : 3258 (chủ ý tạo ra, sẽ điền ở Phần 5)

NaN theo từng cột:
flow_pkts_per_s     1086
bwd_pkts_per_s      1086
flow_bytes_per_s    1086


---
## PHẦN 3 — Khử trùng lặp

Phải làm **trước** khi split. Nếu split trước rồi mới khử, các bản sao của cùng một dòng sẽ nằm cả
ở train lẫn test — mô hình được chấm điểm trên chính những dòng nó đã học thuộc, cho điểm cao giả tạo.

Trùng lặp ở đây là hệ quả tự nhiên của quét cổng: hàng nghìn flow "gửi SYN, nhận RST, 2 gói" khác
nhau đúng ở cổng đích — mà cổng đích đã bị loại khỏi feature (chống rò rỉ định danh). Nên sau khi
bỏ cổng, chúng thành những dòng giống hệt nhau.

In [9]:
# =========================================================
# 3.1 — Khử trùng lặp
# =========================================================
key = FEATURES_V2 + ['label']

n_before   = len(clean)
n_dup      = int(clean.duplicated(subset=key).sum())
conflict   = int((clean.groupby(FEATURES_V2, dropna=False)['label'].nunique() > 1).sum())

dedup = clean.drop_duplicates(subset=key).reset_index(drop=True)

print(f'Trước khử trùng : {n_before}')
print(f'Dòng trùng lặp  : {n_dup} ({n_dup/n_before:.1%})')
print(f'Sau khử trùng   : {len(dedup)}')
print(f'\nNhóm mâu thuẫn nhãn (cùng feature, khác nhãn): {conflict}')
print('  -> Đây là trần lỗi không thể vượt qua: không mô hình nào phân biệt được')
print('     hai dòng có feature giống hệt nhau nhưng nhãn khác nhau.')

print(f'\nPhân bố nhãn sau khử trùng:')
vc = dedup['label'].value_counts()
for k, v in vc.items():
    print(f'  {k:10s}: {v:>5} ({v/len(dedup):6.1%})')

Trước khử trùng : 6024
Dòng trùng lặp  : 1509 (25.0%)
Sau khử trùng   : 4515

Nhóm mâu thuẫn nhãn (cùng feature, khác nhãn): 3
  -> Đây là trần lỗi không thể vượt qua: không mô hình nào phân biệt được
     hai dòng có feature giống hệt nhau nhưng nhãn khác nhau.

Phân bố nhãn sau khử trùng:
  normal    :  2491 ( 55.2%)
  scanport  :  2024 ( 44.8%)


---
## PHẦN 4 — Chia train / val / test

Tỉ lệ **70 / 15 / 15**, phân tầng theo nhãn để mỗi tập giữ đúng tỉ lệ hai lớp.

Cần tập validation **riêng** vì ở notebook train sẽ phải dò siêu tham số (`max_depth`,
`min_samples_leaf`, `ccp_alpha`). Nếu dò trên tập test thì test không còn là ước lượng khách quan
nữa — nó đã tham gia vào quá trình chọn mô hình.

In [10]:
# =========================================================
# 4.1 — Split phân tầng 70/15/15
# =========================================================
LABEL_MAP = {'normal': 0, 'scanport': 1}

X_all = dedup[FEATURES_V2].copy()
y_all = dedup['label'].map(LABEL_MAP)
assert y_all.notna().all(), 'Có nhãn lạ ngoài LABEL_MAP'

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X_all, y_all, test_size=0.30, stratify=y_all, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RANDOM_STATE)

split_report = pd.DataFrame({
    'n':          [len(y_train), len(y_val), len(y_test)],
    'normal':     [int((y == 0).sum()) for y in (y_train, y_val, y_test)],
    'scanport':   [int((y == 1).sum()) for y in (y_train, y_val, y_test)],
}, index=['train', 'val', 'test'])
split_report['tỉ lệ %'] = (split_report['n'] / len(y_all) * 100).round(1)
split_report['%scanport'] = (split_report['scanport'] / split_report['n'] * 100).round(1)

print(split_report.to_string())
print(f'\nTổng: {split_report["n"].sum()} = {len(y_all)} (khớp)')

          n  normal  scanport  tỉ lệ %  %scanport
train  3160    1743      1417     70.0       44.8
val     677     374       303     15.0       44.8
test    678     374       304     15.0       44.8

Tổng: 4515 = 4515 (khớp)


---
## PHẦN 5 — Điền giá trị NaN

Dùng **trung vị** thay vì trung bình, vì các cột tốc độ có đuôi phân phối rất dài — trung bình sẽ
bị vài giá trị cực đoan kéo lệch.

Điểm mấu chốt: **trung vị chỉ được tính trên tập train**, rồi áp cùng giá trị đó cho val và test.
Tính trung vị trên toàn bộ dữ liệu là để thông tin của test rò rỉ ngược vào train.

In [11]:
# =========================================================
# 5.1 — Học trung vị TỪ TRAIN, áp cho cả 3 tập
# =========================================================
impute_values = X_train.median(numeric_only=True)

nan_cols = [f for f in FEATURES_V2 if X_train[f].isna().any() or X_val[f].isna().any() or X_test[f].isna().any()]
print('Trung vị học từ tập train (chỉ các cột có NaN):')
for f in nan_cols:
    print(f'  {f:18s} median(train) = {impute_values[f]:>12,.4f}   '
          f'| NaN: train {int(X_train[f].isna().sum())}, '
          f'val {int(X_val[f].isna().sum())}, test {int(X_test[f].isna().sum())}')

X_train = X_train.fillna(impute_values)
X_val   = X_val.fillna(impute_values)
X_test  = X_test.fillna(impute_values)

print(f'\nNaN còn lại — train: {int(X_train.isna().sum().sum())}, '
      f'val: {int(X_val.isna().sum().sum())}, test: {int(X_test.isna().sum().sum())}')
assert X_train.isna().sum().sum() == 0

print('\nGhi chú: sklearn >= 1.3 cho phép DecisionTree xử lý NaN trực tiếp,')
print('nhưng điền tường minh giúp kết quả tái lập được và dùng chung được với mô hình khác.')

Trung vị học từ tập train (chỉ các cột có NaN):
  flow_pkts_per_s    median(train) =      27.1162   | NaN: train 46, val 15, test 11
  bwd_pkts_per_s     median(train) =      11.7429   | NaN: train 46, val 15, test 11
  flow_bytes_per_s   median(train) =   2,378.8322   | NaN: train 46, val 15, test 11

NaN còn lại — train: 0, val: 0, test: 0

Ghi chú: sklearn >= 1.3 cho phép DecisionTree xử lý NaN trực tiếp,
nhưng điền tường minh giúp kết quả tái lập được và dùng chung được với mô hình khác.


---
## PHẦN 6 — Chuẩn hoá thang đo về [0, 1]

### Vì sao dữ liệu cần chuẩn hoá

Các cột đang ở những thang đo cách nhau rất xa: `flow_bytes_per_s` lên tới **9,5 triệu**, trong khi
`rst_cnt` chỉ nhận 0 hoặc 1. Chuẩn hoá đưa tất cả về cùng khoảng [0, 1] bằng công thức:

$$x_{scaled} = \\frac{x - min_{train}}{max_{train} - min_{train}}$$

### Điều cần biết trước

**Với Decision Tree, bước này không làm thay đổi kết quả.** Cây chia dữ liệu bằng câu hỏi dạng
`x <= ngưỡng`; phép co giãn tuyến tính chỉ dời ngưỡng từ `60` sang `0.0005` chứ không đổi *thứ tự*
các mẫu, nên cây mọc ra y hệt và dự đoán trùng khít từng dòng. Ô 6.2 kiểm chứng lại điều này bằng
số thay vì bắt bạn tin.

Lợi ích thật của bước này nằm ở chỗ khác:

- **Mở đường cho mô hình khác.** KNN, SVM, hồi quy logistic và mạng nơ-ron đều tính khoảng cách
  hoặc nhân trọng số trực tiếp trên giá trị thô. Không chuẩn hoá thì `flow_bytes_per_s` với thang
  hàng triệu sẽ nuốt chửng mọi cột khác.
- **Dễ so sánh giữa các cột** khi vẽ biểu đồ hay xem phân phối.

### Nguyên tắc bắt buộc

`min` và `max` **chỉ được lấy từ tập train**, giống hệt quy tắc đã dùng cho bước điền NaN. Lấy
min/max trên toàn bộ dữ liệu là để tập test rò rỉ vào quá trình huấn luyện.

Hệ quả: val và test có thể chứa giá trị lớn hơn `max` của train, khi đó kết quả vượt quá 1. Đây là
**hành vi đúng**, không phải lỗi — nó phản ánh trung thực rằng dữ liệu mới có giá trị nằm ngoài
những gì mô hình từng thấy. Ô 6.1 đếm chính xác bao nhiêu trường hợp như vậy.

In [12]:
# =========================================================
# 6.1 — Học min/max TỪ TRAIN, áp cho cả 3 tập
# =========================================================
from sklearn.preprocessing import MinMaxScaler

CLIP_TO_RANGE = False   # True = ép val/test về đúng [0,1]; False = giữ nguyên giá trị thật

scaler = MinMaxScaler(feature_range=(0, 1)).fit(X_train)

X_train_raw = X_train.copy()          # giữ bản thô để ô 6.2 đối chiếu
X_val_raw, X_test_raw = X_val.copy(), X_test.copy()

def apply_scale(df_):
    out = pd.DataFrame(scaler.transform(df_), columns=FEATURES_V2, index=df_.index)
    return out.clip(0, 1) if CLIP_TO_RANGE else out

X_train, X_val, X_test = apply_scale(X_train_raw), apply_scale(X_val_raw), apply_scale(X_test_raw)

print(f'Đã chuẩn hoá {len(FEATURES_V2)} cột về [0, 1]  (cắt về đúng khoảng: {CLIP_TO_RANGE})\n')
print(f'Tập train : min={X_train.to_numpy().min():.4f}  max={X_train.to_numpy().max():.4f}   '
      f'<- luôn đúng [0,1] vì min/max học từ chính tập này')

for name, Xs in [('val', X_val), ('test', X_test)]:
    arr = Xs.to_numpy()
    n_out = int(((arr < 0) | (arr > 1)).sum())
    print(f'Tập {name:5s}: min={arr.min():.4f}  max={arr.max():.4f}   '
          f'<- {n_out} giá trị ngoài [0,1] ({n_out/arr.size:.3%})')

outside = {}
for c in FEATURES_V2:
    n = int(((X_val[c] > 1) | (X_val[c] < 0)).sum() + ((X_test[c] > 1) | (X_test[c] < 0)).sum())
    if n:
        outside[c] = n
if outside:
    print('\nCột có giá trị vượt khoảng train (giá trị mới lớn hơn mọi thứ từng thấy lúc train):')
    for c, n in sorted(outside.items(), key=lambda kv: -kv[1]):
        print(f'  {c:22s} {n:>3} giá trị')

Đã chuẩn hoá 24 cột về [0, 1]  (cắt về đúng khoảng: False)

Tập train : min=0.0000  max=1.0000   <- luôn đúng [0,1] vì min/max học từ chính tập này
Tập val  : min=0.0000  max=1.2000   <- 1 giá trị ngoài [0,1] (0.006%)
Tập test : min=0.0000  max=1.2000   <- 5 giá trị ngoài [0,1] (0.031%)

Cột có giá trị vượt khoảng train (giá trị mới lớn hơn mọi thứ từng thấy lúc train):
  syn_cnt                  3 giá trị
  fwd_iat_std              1 giá trị
  bwd_iat_mean             1 giá trị
  idle_std                 1 giá trị


In [13]:
# =========================================================
# 6.2 — Kiểm chứng: chuẩn hoá có làm đổi kết quả không?
# =========================================================
from sklearn.tree import DecisionTreeClassifier

_p = dict(max_depth=8, class_weight='balanced', random_state=RANDOM_STATE)
_a = DecisionTreeClassifier(**_p).fit(X_train_raw, y_train)   # dữ liệu thô
_b = DecisionTreeClassifier(**_p).fit(X_train,     y_train)   # dữ liệu đã chuẩn hoá

_pa, _pb = _a.predict(X_test_raw), _b.predict(X_test)

print(f'Cây học trên dữ liệu THÔ        : {_a.get_n_leaves()} lá, '
      f'accuracy trên test = {(_pa == y_test).mean():.6f}')
print(f'Cây học trên dữ liệu ĐÃ CHUẨN HOÁ: {_b.get_n_leaves()} lá, '
      f'accuracy trên test = {(_pb == y_test).mean():.6f}')
print(f'\nSố dòng hai bên dự đoán khác nhau: {int((_pa != _pb).sum())} / {len(_pa)}')
print('\n=> Giống hệt nhau. Với Decision Tree, chuẩn hoá không cải thiện cũng không làm hỏng gì.')
print('   Bước này là để dành cho các mô hình nhạy thang đo dùng sau này.')

Cây học trên dữ liệu THÔ        : 39 lá, accuracy trên test = 0.988201
Cây học trên dữ liệu ĐÃ CHUẨN HOÁ: 39 lá, accuracy trên test = 0.988201

Số dòng hai bên dự đoán khác nhau: 0 / 678

=> Giống hệt nhau. Với Decision Tree, chuẩn hoá không cải thiện cũng không làm hỏng gì.
   Bước này là để dành cho các mô hình nhạy thang đo dùng sau này.


In [14]:
# =========================================================
# 6.3 — Xuất tham số chuẩn hoá ra CSV
# =========================================================
# Không có file này thì không thể dùng mô hình ở nơi khác: dữ liệu mới bắt buộc
# phải được chuẩn hoá bằng ĐÚNG những con số dưới đây, không phải min/max của chính nó.
#
# Hai cột theo đúng công thức triển khai quen thuộc:  x_scaled = (x - mean) / scale
# Với MinMaxScaler về [0,1] thì cặp đó chính là:
#     mean  = data_min_    (giá trị nhỏ nhất của tập train)
#     scale = data_range_  (= data_max - data_min)
# LƯU Ý: 'mean' ở đây KHÔNG phải trung bình cộng. Dùng trung bình cộng vào công thức
# này sẽ cho ra một phép biến đổi khác hẳn, không tái tạo được dữ liệu đã train.
scaler_df = pd.DataFrame({
    'feature': FEATURES_V2,
    'mean':    scaler.data_min_,
    'scale':   scaler.data_range_,
})

SCALER_CSV = EXPORT_DIR / 'scaler_params.csv'      # đi cùng mô hình, không nằm ở dataset_csv
scaler_df.to_csv(SCALER_CSV, index=False, encoding='utf-8-sig')

print(f'Đã lưu: {SCALER_CSV}\n')
print(f'Công thức dùng lại:  x_scaled = (x - mean) / scale\n')
print(scaler_df.round(6).to_string(index=False))

# đối chiếu: áp công thức 2 cột lên toàn bộ tập train, phải trùng khít kết quả thật
_check = (X_train_raw - scaler_df['mean'].values) / scaler_df['scale'].values
assert np.allclose(_check.values, X_train.values), 'Hai cột mean/scale không tái tạo đúng dữ liệu'
print(f'\nĐã đối chiếu trên toàn bộ {len(X_train)} dòng tập train — công thức khớp tuyệt đối.')

print('\nCác tham số khác (min_, scale_ của sklearn, giá trị điền NaN) nằm trong')
print('preprocessing_meta.json, không đưa vào CSV này cho gọn.')

Đã lưu: D:\01.AI_Security\Camera_AI_ONT\Decesion_trees_train_model\model_export\scaler_params.csv

Công thức dùng lại:  x_scaled = (x - mean) / scale

              feature      mean        scale
      flow_pkts_per_s  0.034929 1.978442e+04
       bwd_pkts_per_s  0.000000 9.892226e+03
     flow_bytes_per_s  2.924199 9.549676e+06
             tot_pkts  1.000000 5.846050e+05
         flow_iat_min  0.000000 8.532505e+00
        flow_iat_mean  0.000000 4.294433e+01
          fwd_iat_min  0.000000 9.628418e+01
         fwd_iat_mean  0.000000 9.628418e+01
          fwd_iat_std  0.000000 4.873434e+01
         bwd_iat_mean  0.000000 8.014396e+01
             idle_std  0.000000 3.559366e+01
          pkt_len_min 42.000000 1.472000e+03
          pkt_len_max 42.000000 3.477600e+04
         pkt_len_mean 42.000000 2.142565e+03
          pkt_len_std  0.000000 6.726211e+03
      fwd_pkt_len_std  0.000000 1.131035e+04
     bwd_pkt_len_mean  0.000000 1.494000e+03
              syn_cnt  0.000000 5.00000

---
## PHẦN 7 — Cân bằng lớp

### Kiểm tra trước: dữ liệu có thật sự mất cân bằng không?

Trước khi gọi SMOTE theo phản xạ, cần đo đã. Việc gán lại nhãn ở notebook trước đã chuyển 1.277
flow traffic nền từ `scanport` sang `normal`, làm tỉ lệ hai lớp thay đổi rất nhiều so với ban đầu.

In [15]:
# =========================================================
# 7.1 — Đo mức mất cân bằng
# =========================================================
cnt = y_train.value_counts().sort_index()
ratio = cnt.max() / cnt.min()

print(f'Tập train — normal: {cnt[0]}, scanport: {cnt[1]}')
print(f'Tỉ lệ mất cân bằng: 1 : {ratio:.2f}\n')

if ratio < 1.5:
    verdict = 'CÂN BẰNG — không cần lấy mẫu lại'
elif ratio < 4:
    verdict = 'LỆCH NHẸ — class_weight là đủ'
elif ratio < 10:
    verdict = 'LỆCH VỪA — nên cân nhắc lấy mẫu lại'
else:
    verdict = 'LỆCH NẶNG — bắt buộc xử lý'
print(f'Đánh giá: {verdict}')

print(f'\nĐể so sánh — tỉ lệ trước khi gán lại nhãn ở notebook trước: 1 : 2.34')
print('Việc sửa nhãn đã tự nó giải quyết phần lớn vấn đề mất cân bằng.')

Tập train — normal: 1743, scanport: 1417
Tỉ lệ mất cân bằng: 1 : 1.23

Đánh giá: CÂN BẰNG — không cần lấy mẫu lại

Để so sánh — tỉ lệ trước khi gán lại nhãn ở notebook trước: 1 : 2.34
Việc sửa nhãn đã tự nó giải quyết phần lớn vấn đề mất cân bằng.


### So sánh 5 chiến lược

Vẫn thử đủ 5 phương án và **đo bằng số** thay vì tin vào cảm tính. Hai điểm kỹ thuật quan trọng:

- Dùng `imblearn.pipeline.Pipeline` chứ không phải `sklearn.pipeline.Pipeline`: chỉ bản của imblearn
  mới đảm bảo bước lấy mẫu lại **chỉ chạy trên phần train của mỗi fold**, không đụng vào phần
  validation. Lấy mẫu lại trước khi chia fold là một trong những lỗi rò rỉ phổ biến nhất khi dùng SMOTE.
- Dùng `RepeatedStratifiedKFold` (5 fold × 4 lần lặp = 20 phép đo) để có **độ lệch chuẩn**. Nếu không
  có sai số thì không thể biết chênh lệch giữa hai chiến lược là thật hay chỉ là nhiễu.

In [16]:
# =========================================================
# 7.2 — So sánh có sai số
# =========================================================
def make_tree():
    return DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE)

strategies = {
    'Không làm gì':        SkPipeline([('clf', make_tree())]),
    'class_weight':        SkPipeline([('clf', DecisionTreeClassifier(
                               max_depth=8, class_weight='balanced', random_state=RANDOM_STATE))]),
    'RandomUnderSampler':  ImbPipeline([('res', RandomUnderSampler(random_state=RANDOM_STATE)),
                                        ('clf', make_tree())]),
    'RandomOverSampler':   ImbPipeline([('res', RandomOverSampler(random_state=RANDOM_STATE)),
                                        ('clf', make_tree())]),
    'SMOTE':               ImbPipeline([('res', SMOTE(random_state=RANDOM_STATE)),
                                        ('clf', make_tree())]),
}

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=RANDOM_STATE)

print('Đánh giá trên tập train, 5 fold x 4 lần lặp = 20 phép đo\n')
results = []
for name, pipe in strategies.items():
    s = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    results.append({'chiến lược': name, 'f1_macro': s.mean(), 'std': s.std(),
                    'min': s.min(), 'max': s.max()})
    print(f'  {name:20s} f1_macro = {s.mean():.4f} +- {s.std():.4f}')

res = pd.DataFrame(results).sort_values('f1_macro', ascending=False).reset_index(drop=True)

spread = res['f1_macro'].max() - res['f1_macro'].min()
noise  = res['std'].mean()
print(f'\nChênh lệch giữa tốt nhất và kém nhất : {spread:.4f}')
print(f'Độ lệch chuẩn trung bình (mức nhiễu)  : {noise:.4f}')
print(f'=> Chênh lệch {"NHỎ HƠN" if spread < noise else "LỚN HƠN"} mức nhiễu '
      f'-> khác biệt {"KHÔNG có ý nghĩa thống kê" if spread < noise else "CÓ ý nghĩa"}.')

Đánh giá trên tập train, 5 fold x 4 lần lặp = 20 phép đo



  Không làm gì         f1_macro = 0.9877 +- 0.0035

  class_weight         f1_macro = 0.9875 +- 0.0036


  RandomUnderSampler   f1_macro = 0.9868 +- 0.0041
  RandomOverSampler    f1_macro = 0.9883 +- 0.0038


  SMOTE                f1_macro = 0.9889 +- 0.0042

Chênh lệch giữa tốt nhất và kém nhất : 0.0021
Độ lệch chuẩn trung bình (mức nhiễu)  : 0.0038
=> Chênh lệch NHỎ HƠN mức nhiễu -> khác biệt KHÔNG có ý nghĩa thống kê.


In [17]:
# =========================================================
# 7.3 — Kiểm tra tác dụng phụ của SMOTE trên cột dạng đếm
# =========================================================
# SMOTE nội suy tuyến tính giữa 2 mẫu lân cận -> về lý thuyết có thể sinh
# giá trị kiểu syn_cnt = 2.7, vô nghĩa với một biến đếm số gói.
INT_COLS = [f for f in FEATURES_V2 if (X_train[f].dropna() % 1 == 0).all()]
print(f'Cột mang giá trị nguyên: {len(INT_COLS)}')
print(f'  {INT_COLS}\n')

X_sm, y_sm = SMOTE(random_state=RANDOM_STATE).fit_resample(X_train, y_train)
frac = {c: int((X_sm[c] % 1 != 0).sum()) for c in INT_COLS if (X_sm[c] % 1 != 0).any()}

if frac:
    print('SMOTE đã sinh giá trị không nguyên:')
    for c, n in frac.items():
        print(f'  {c:22s} {n} giá trị -> cần làm tròn nếu dùng SMOTE')
else:
    print('SMOTE KHÔNG sinh giá trị không nguyên trên dataset này.')
    print('Lý do: các cột đếm ở đây có rất ít giá trị phân biệt (rst_cnt chỉ có 0/1,')
    print('syn_cnt có 7 giá trị), nên các mẫu lân cận thường trùng giá trị,')
    print('phép nội suy trả về đúng giá trị đó. Rủi ro là có thật nhưng chưa xảy ra ở đây.')

Cột mang giá trị nguyên: 1
  ['rst_cnt']

SMOTE KHÔNG sinh giá trị không nguyên trên dataset này.
Lý do: các cột đếm ở đây có rất ít giá trị phân biệt (rst_cnt chỉ có 0/1,
syn_cnt có 7 giá trị), nên các mẫu lân cận thường trùng giá trị,
phép nội suy trả về đúng giá trị đó. Rủi ro là có thật nhưng chưa xảy ra ở đây.


### Kết luận

Cả năm chiến lược cho kết quả nằm gọn trong biên độ nhiễu của nhau: chênh lệch giữa phương án cao
nhất và thấp nhất còn **nhỏ hơn độ lệch chuẩn** của chính các phép đo. Điều này hợp lý — dữ liệu đã
gần cân bằng sẵn (1 : 1,23), nên lấy mẫu lại chỉ thêm bản sao hoặc mẫu tổng hợp mà không bổ sung
thông tin nào mới.

SMOTE tình cờ đứng đầu bảng xếp hạng, nhưng **khoảng cách đó không có ý nghĩa thống kê** — chạy lại
với `random_state` khác thì thứ tự hoàn toàn có thể đảo. Chọn theo thứ hạng trong trường hợp này là
chọn theo nhiễu.

**Chọn `class_weight='balanced'`** làm phương án mặc định:

- Hiệu năng ngang các phương án khác nhưng **không đụng vào dữ liệu** — không nhân bản, không sinh
  mẫu giả, tập train giữ nguyên phân phối thật
- Không tốn thêm bộ nhớ hay thời gian
- Tự động thích ứng nếu sau này bổ sung dữ liệu tấn công làm tỉ lệ lệch đi

Nghĩa là **không lấy mẫu lại**, chỉ đặt trọng số lớp lúc train. Vì vậy notebook lưu ra tập train
nguyên bản; việc cân bằng nằm ở tham số của mô hình chứ không nằm trong dữ liệu.

In [18]:
# =========================================================
# 7.4 — Chốt phương án
# =========================================================
BALANCE_STRATEGY = 'class_weight'     # 'class_weight' | 'none' | 'undersample' | 'oversample' | 'smote'

print('Bảng xếp hạng:')
print(res.round(4).to_string(index=False))
print(f'\nPhương án chọn: {BALANCE_STRATEGY}')
print('-> Dữ liệu train được lưu NGUYÊN BẢN, không lấy mẫu lại.')
print("-> Ở notebook train, đặt: DecisionTreeClassifier(class_weight='balanced', ...)")

from sklearn.utils.class_weight import compute_class_weight
classes = np.array([0, 1])
weights = compute_class_weight('balanced', classes=classes, y=y_train)
CLASS_WEIGHT = {int(c): round(float(w), 6) for c, w in zip(classes, weights)}
print(f'\nTrọng số tương ứng: {CLASS_WEIGHT}')
print('  (normal = 0, scanport = 1)')

Bảng xếp hạng:
        chiến lược  f1_macro    std    min    max
             SMOTE    0.9889 0.0042 0.9776 0.9968
 RandomOverSampler    0.9883 0.0038 0.9808 0.9968
      Không làm gì    0.9877 0.0035 0.9825 0.9968
      class_weight    0.9875 0.0036 0.9777 0.9952
RandomUnderSampler    0.9868 0.0041 0.9792 0.9952

Phương án chọn: class_weight
-> Dữ liệu train được lưu NGUYÊN BẢN, không lấy mẫu lại.
-> Ở notebook train, đặt: DecisionTreeClassifier(class_weight='balanced', ...)

Trọng số tương ứng: {0: 0.906483, 1: 1.115032}
  (normal = 0, scanport = 1)


---
## PHẦN 8 — Lưu kết quả

In [19]:
# =========================================================
# 8.1 — Gộp 3 tập vào một file, phân biệt bằng cột `split`
# =========================================================
parts = []
for name, Xp, yp in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
    part = Xp.copy()
    part['label'] = yp.values
    part['split'] = name
    parts.append(part)

ready = pd.concat(parts, ignore_index=False).sort_index()
ready = ready[['split'] + FEATURES_V2 + ['label']]
ready.to_csv(OUT_CSV, index=False)

print(f'Đã lưu: {OUT_CSV}')
print(f'  {len(ready)} dòng x {ready.shape[1]} cột')
print(f'  {len(FEATURES_V2)} feature + cột split + cột label (0=normal, 1=scanport)\n')
print(pd.crosstab(ready['split'], ready['label'],
                  rownames=['split'], colnames=['label']).to_string())

Đã lưu: D:\01.AI_Security\Camera_AI_ONT\dataset_csv\ids_dataset_ready.csv
  4515 dòng x 26 cột
  24 feature + cột split + cột label (0=normal, 1=scanport)

label     0     1
split            
test    374   304
train  1743  1417
val     374   303


In [20]:
# =========================================================
# 8.2 — Lưu metadata để notebook train tái lập chính xác
# =========================================================
preprocessing_meta = {
    'source_file': IN_CSV.name,
    'output_file': OUT_CSV.name,
    'features': FEATURES_V2,
    'n_features': len(FEATURES_V2),
    'label_map': LABEL_MAP,
    'random_state': RANDOM_STATE,

    'cleaning': {
        'div_guard_constant': DIV_GUARD,
        'rate_features': RATE_FEATURES,
        'physical_limits': {'max_bytes_per_s': MAX_BYTES_PER_S, 'max_pkts_per_s': MAX_PKTS_PER_S},
        'inf_cells_to_nan': log['inf_to_nan'],
        'undefined_rate_flows': int(is_pseudo_inf.sum()),
        'undefined_rate_by_guard': int(is_guard.sum()),
        'undefined_rate_by_physical_limit': int((is_impossible & ~is_guard).sum()),
        'undefined_rate_cells_to_nan': log['pseudo_inf_to_nan'],
        'negative_cells_clipped': log['negative_clipped'],
        'added_flag_column': 'zero_duration_flow' if ADD_ZERO_DURATION_FLAG else None,
    },

    'deduplication': {'before': n_before, 'removed': n_dup, 'after': len(dedup),
                      'conflicting_groups': conflict},

    'split': {'ratio': '70/15/15', 'stratified': True,
              'train': len(y_train), 'val': len(y_val), 'test': len(y_test)},

    'imputation': {'method': 'median', 'fitted_on': 'train_only',
                   'values': {k: float(v) for k, v in impute_values.items()}},

    'scaling': {
        'method': 'MinMaxScaler',
        'feature_range': [0, 1],
        'fitted_on': 'train_only',
        'clipped': CLIP_TO_RANGE,
        'params_csv': f'model_export/{SCALER_CSV.name}',
        'formula': 'x_scaled = (x - data_min) / data_range  =  x * scale_ + min_',
        'data_min':   {f: float(v) for f, v in zip(FEATURES_V2, scaler.data_min_)},
        'data_range': {f: float(v) for f, v in zip(FEATURES_V2, scaler.data_range_)},
        'scale_':     {f: float(v) for f, v in zip(FEATURES_V2, scaler.scale_)},
        'min_':       {f: float(v) for f, v in zip(FEATURES_V2, scaler.min_)},
        'note': ('Giá trị của val/test có thể vượt ra ngoài [0,1] khi dữ liệu mới lớn hơn '
                 'khoảng đã thấy lúc train. Đó là hành vi đúng, không phải lỗi.'),
    },

    'balancing': {
        'imbalance_ratio': round(float(ratio), 4),
        'verdict': verdict,
        'strategy': BALANCE_STRATEGY,
        'class_weight': CLASS_WEIGHT,
        'comparison': res.round(6).to_dict('records'),
    },
}

META_OUT.write_text(json.dumps(preprocessing_meta, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Đã lưu metadata: {META_OUT}')
print(json.dumps({k: (v if not isinstance(v, dict) else '...')
                  for k, v in preprocessing_meta.items()}, indent=2, ensure_ascii=False))

Đã lưu metadata: D:\01.AI_Security\Camera_AI_ONT\dataset_csv\preprocessing_meta.json
{
  "source_file": "ids_dataset_labeled.csv",
  "output_file": "ids_dataset_ready.csv",
  "features": [
    "flow_pkts_per_s",
    "bwd_pkts_per_s",
    "flow_bytes_per_s",
    "tot_pkts",
    "flow_iat_min",
    "flow_iat_mean",
    "fwd_iat_min",
    "fwd_iat_mean",
    "fwd_iat_std",
    "bwd_iat_mean",
    "idle_std",
    "pkt_len_min",
    "pkt_len_max",
    "pkt_len_mean",
    "pkt_len_std",
    "fwd_pkt_len_std",
    "bwd_pkt_len_mean",
    "syn_cnt",
    "rst_cnt",
    "fin_cnt",
    "down_up_pkt_ratio",
    "down_up_byte_ratio",
    "bwd_payload_bytes",
    "bwd_pkts_with_payload"
  ],
  "n_features": 24,
  "label_map": "...",
  "random_state": 42,
  "cleaning": "...",
  "deduplication": "...",
  "split": "...",
  "imputation": "...",
  "scaling": "...",
  "balancing": "..."
}


---
## Tóm tắt và lưu ý

### Đã làm

| Bước | Kết quả |
|---|---|
| Kiểm tra số học | 0 NaN, 0 Inf theo nghĩa đen — nhưng phát hiện **1.086 flow mang tốc độ giả vô cực** (1.075 do hằng số chặn chia 0, thêm 11 do duration dưới 1 micro-giây) |
| Làm sạch | Tốc độ không xác định → `NaN` (max `flow_bytes_per_s` giảm từ 4,19 GB/s xuống 9,5 MB/s); 10 giá trị IAT âm → 0 |
| Số feature | Giữ nguyên **24** như đã chốt từ Excel — cột cờ `zero_duration_flow` được đo và loại vì trùng lặp thông tin với `tot_pkts` |
| Khử trùng lặp | Bỏ ~25% dòng trùng, **trước** khi split |
| Split | 70/15/15 phân tầng, có tập validation riêng để dò siêu tham số |
| Điền NaN | Trung vị, **chỉ học từ train** |
| Chuẩn hoá | MinMaxScaler về [0, 1], min/max **chỉ học từ train**, tham số xuất ra `scaler_params.csv` |
| Cân bằng | So sánh 5 chiến lược trên 20 phép đo → chọn `class_weight='balanced'` |

### Điều cần nói thẳng

Với **Decision Tree trên dataset này**, các thao tác trên gần như **không làm thay đổi điểm số**
(f1_macro dao động quanh 0,990 ở mọi phương án). Đó không phải là lý do để bỏ qua chúng:

1. **Phơi bày lỗi dữ liệu thật.** Giá trị 4,19 GB/s trên mạng LAN là bất khả thi. Không phát hiện
   ra thì nó sẽ âm thầm đi vào mọi phân tích về sau.
2. **Cần cho lần capture sau.** Dataset hiện tại tình cờ không có NaN/Inf; lần bắt gói tiếp theo
   thì chưa chắc. Code đã sẵn sàng.
3. **Cần cho mô hình khác.** Decision Tree bất biến với biến đổi đơn điệu nên miễn nhiễm với giá
   trị cực đoan. Hồi quy logistic, KNN hay mạng nơ-ron thì không — với chúng, giá trị 4 tỉ sẽ chi
   phối toàn bộ.
4. **Thứ tự các bước mới là thứ thật sự quan trọng.** Khử trùng lặp trước split, điền NaN và cân
   bằng sau split và chỉ trên train — làm sai thứ tự sẽ cho điểm cao giả tạo. Đây mới là phần dễ
   sai và tốn kém nhất nếu mắc phải.

### Còn lại

Điểm số cao hiện nay vẫn phản ánh **một phiên nmap duy nhất**. Việc quan trọng nhất còn lại không
nằm ở tiền xử lý mà ở dữ liệu: capture thêm các phiên scan đa dạng (`-T2` chậm, UDP scan, attacker
khác, target khác) và dữ liệu brute-force. Trước khi có chúng, chưa thể kết luận gì về khả năng
tổng quát hoá.

### File xuất ra

| File | Nội dung |
|---|---|
| `ids_dataset_ready.csv` | Dữ liệu đã chuẩn hoá về [0, 1], kèm cột `split` và `label` |
| `model_export/scaler_params.csv` | Hai cột `mean` và `scale` cho từng feature, dùng theo công thức `x_scaled = (x - mean) / scale` |
| `preprocessing_meta.json` | Toàn bộ cấu hình để tái lập chính xác |

File này được ghi thẳng vào `model_export/` chứ không để trong `dataset_csv/`, vì nó **bắt buộc
phải đi kèm mô hình** khi triển khai chứ không phải sản phẩm trung gian của khâu xử lý dữ liệu. Dữ liệu mới phải được chuẩn
hoá bằng đúng những con số trong đó — không phải bằng min/max của chính lô dữ liệu mới. Tính lại
min/max trên dữ liệu mới là một lỗi kín đáo: mô hình vẫn chạy, vẫn trả kết quả, chỉ là sai.

**Tiếp theo:** `model_train.ipynb` — đọc `ids_dataset_ready.csv` + `preprocessing_meta.json`, dò
siêu tham số trên tập val, đánh giá lần cuối trên test.